# 🔗 Chapter 6: Algorithm Chains and Pipelines
**Book Reference:** *Introduction to Machine Learning with Python: A Guide for Data Scientists* by Andreas C. Müller & Sarah Guido

---

## 6.1 The Data Leakage Catastrophe in Model Evaluation
In typical machine learning applications, data preprocessing is a multi-step chain. For example, raw datasets must be scaled (e.g., using `MinMaxScaler`) before being passed into a supervised classifier like a Support Vector Machine (`SVC`). 

A major trap when evaluating models using cross-validation or simple splits is performing preprocessing steps outside the validation loop. Consider the following workflow:
1. Fit a scaling transformer on the entire dataset.
2. Transform the entire dataset using those computed parameters.
3. Split the scaled data into training and testing partitions.
4. Train a model and score its performance on the testing subset.

### Why this is fundamentally broken:
When you fit a scaler across the entire dataset, the scaler observes the minimum, maximum, mean, and standard deviation values of the testing partition before the split occurs. Information from the test set leaks directly into the training process. This is known as **Data Leakage** or **Information Leakage**.

Data leakage leads to overly optimistic performance metrics during validation. When the model is deployed to production and handles truly unseen records, its performance often drops significantly because the initial evaluation was compromised. To prevent data leakage, preprocessing parameters must be calculated *only* from the training data partition.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline, make_pipeline

# Ensure consistent, reproducible environments
np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

print("Ecosystem modules successfully loaded for Chapter 6 Pipeline exploration.")

### 6.1.1 Demonstrating the Naive Way (With Data Leakage)
Let's replicate the book's demonstration of a flawed parameter tuning loop. We will intentionally leak testing information into the scaling phase to see how it creates an overly optimistic validation score.

In [ ]:
cancer = load_breast_cancer()

# Naive Approach: Scale the entire dataset FIRST before splitting or using cross-validation
scaler = MinMaxScaler()
X_scaled_naive = scaler.fit_transform(cancer.data)

# Perform the split on already scaled data (Information has leaked!)
X_train_naive, X_test_naive, y_train_naive, y_test_naive = train_test_split(
    X_scaled_naive, cancer.target, test_size=0.3, random_state=0
)

# Tune hyperparameters using a grid search over the leaked dataset
param_grid = {'C': [0.001, 0.01, 0.1, 1, 10, 100], 'gamma': [0.001, 0.01, 0.1, 1, 10, 100]}
naive_grid = GridSearchCV(SVC(), param_grid, cv=5)
naive_grid.fit(X_train_naive, y_train_naive)

print(f"Optimistic Best Cross-Validation Accuracy Score: {naive_grid.best_score_:.4f}")
print(f"Naive Test Evaluation Score:                    {naive_grid.score(X_test_naive, y_test_naive):.4f}")

## 6.2 Building Automated Chains using the Pipeline Interface
To eliminate data leakage, the `Pipeline` class in `scikit-learn` packages multiple processing transformations alongside a terminal estimator into a single unified object.

### Architectural Mechanics of a Pipeline Wrapper
A pipeline is defined as a series of steps, where each step is a string name paired with an object instance. 

* **Mid-Chain Requirements:** Every step before the final step must be a **Transformer** (meaning it implements both `.fit()` and `.transform()` methods).
* **Terminal Step Requirement:** The final step must be an **Estimator** (such as a Classifier or Regressor).

When you invoke `pipeline.fit(X_train, y_train)`, the pipeline runs a sequential execution loop behind the scenes:
1. It takes Step 1, runs `.fit_transform()` on `X_train`, and generates an updated data matrix.
2. It passes that updated data matrix to Step 2, runs `.fit_transform()`, and passes the output down the chain.
3. It passes the final transformed data matrix to the terminal estimator's `.fit()` method.

```
[ Raw Input ] ──> (Step 1: Scaler) ──> (Step 2: PCA) ──> [ Final Estimator Model ]
```

When you call `pipeline.predict(X_test)`, the pipeline automatically restricts its operations to prevent data leakage. It calls `.transform()` on each mid-chain step sequentially using the parameters stored during training, without recalculating any new distributions from the test data.

In [ ]:
# Re-split raw data to establish a clean, uncompromised baseline pipeline
X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, test_size=0.3, random_state=0
)

# Define an explicit pipeline using named structural tuple steps
pipe = Pipeline([
    ('scaler', MinMaxScaler()),
    ('svm', SVC())
])

# Fitting the pipeline automatically scales the data and trains the model safely
pipe.fit(X_train, y_train)

print(f"Unbiased Test Evaluation Accuracy Score: {pipe.score(X_test, y_test):.4f}")

### 6.2.1 Alternative Instantiation via `make_pipeline`
If you don't need to explicitly name each pipeline step, you can use the shorthand helper function `make_pipeline`. This automatically generates lower-case string names for each step based on its class name (e.g., an instance of `StandardScaler` becomes a step named `'standardscaler'`).

In [ ]:
# Construct an identical pipeline using the make_pipeline convenience wrapper
shorthand_pipe = make_pipeline(StandardScaler(), SVC())

print("Automatically generated pipeline step names:")
print([step_name for step_name in shorthand_pipe.named_steps.keys()])

## 6.3 Integrating Pipelines inside Grid Searches
Using a pipeline inside a grid search ensures that the scaling transformations are calculated strictly within each fold of the cross-validation loop. For every individual cross-validation split, the data is scaled from scratch using only the training folds before evaluating the validation fold.

### Parameter Grid Syntax for Pipelines
To modify hyperparameters of steps inside a pipeline, you must use a specific naming convention in the parameter grid dictionary. The key string is written as the **name of the step**, followed by a **double underscore (`__`)**, and then the **name of the parameter**.

* Example: To tune the parameter `C` inside a step named `'svm'`, the parameter grid key must be written as `'svm__C'`.

In [ ]:
# Replicate the book's proper grid search pipeline orchestration
pipe_setup = Pipeline([
    ('scaler', MinMaxScaler()),
    ('svm', SVC())
])

# Set up the parameter grid using the double underscore step syntax
pipeline_param_grid = {
    'svm__C': [0.001, 0.01, 0.1, 1, 10, 100],
    'svm__gamma': [0.001, 0.01, 0.1, 1, 10, 100]
}

# Pass the complete pipeline directly into GridSearchCV
grid_pipeline = GridSearchCV(pipe_setup, param_grid=pipeline_param_grid, cv=5)
grid_pipeline.fit(X_train, y_train)

print(f"Unbiased Cross-Validation Best Score: {grid_pipeline.best_score_:.4f}")
print(f"Optimal Parameter Configurations:     {grid_pipeline.best_params_}")
print(f"True Generalization Test Score:       {grid_pipeline.score(X_test, y_test):.4f}")

## 6.4 Advanced Pipeline Design: Grid Searching Preprocessing Operations
Pipelines can also be used to search over alternative preprocessing architectures. For example, you can use a grid search to decide whether `StandardScaler` or `MinMaxScaler` yields better performance, or even test if a feature reduction step like Principal Component Analysis (`PCA`) should be included.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Construct a dynamic pipeline placeholder assembly
flexible_pipeline = Pipeline([
    ('preprocessing', StandardScaler()),  # Placeholder scaler step
    ('classifier', SVC())                 # Placeholder estimator step
])

# Define an advanced search grid with distinct structural routing paths
advanced_param_grid = [
    {
        'preprocessing': [StandardScaler(), MinMaxScaler()],
        'classifier': [SVC()],
        'classifier__C': [0.1, 1, 10],
        'classifier__gamma': [0.01, 0.1, 1]
    },
    {
        'preprocessing': [None],  # Tree ensembles do not require numerical scaling transformations
        'classifier': [RandomForestClassifier()],
        'classifier__n_estimators': [50, 100, 150],
        'classifier__max_features': [1, 2, 3]
    }
]

advanced_search = GridSearchCV(flexible_pipeline, advanced_param_grid, cv=5)
advanced_search.fit(X_train, y_train)

print(f"Advanced Routing Search Best Parameters Found:\n{advanced_search.best_params_}")
print(f"\nFinal Validation Test Score achieved: {advanced_search.score(X_test, y_test):.4f}")

### 6.5 Summary of Key Takeaways
Chapter 6 emphasizes that maintaining a clean validation boundary across multi-step modeling workflows is critical for machine learning engineering.

**Core Takeaways:**
1. **Data Leakage is Dangerous:** Applying preprocessing transformations like scaling or feature reduction across a complete dataset before validation leaks test data information into the model, leading to overly optimistic evaluation scores.
2. **Encapsulation:** The `Pipeline` utility bundles sequential transformations and a final estimator into a single object, ensuring validation boundaries remain secure.
3. **Cross-Validation Integration:** Passing pipelines into automated validation loops like `GridSearchCV` ensures that data transformations are refitted strictly on the training folds of every cross-validation split.
4. **Flexible Parameter Search:** The double underscore syntax (`step__parameter`) allows you to tune both the hyperparameters of the final model and alternative preprocessing steps in a single grid search.